# Kaggle Offline Train And Submit

This notebook is a single offline Kaggle workflow.

It does four things in one pass:
1. loads the cleaned training dataset and the offline base model
2. runs one QLoRA fine-tuning pass
3. reloads the base model, merges the adapter, and runs deterministic inference
4. writes `/kaggle/working/submission.csv` and `/kaggle/working/submission_debug.csv`

Fixed Kaggle inputs:
- `/kaggle/input/svg-kaggle-data/test.csv`
- `/kaggle/input/svg-kaggle-data/sample_submission.csv`
- `/kaggle/input/svg-kaggle-data/datasets/train_canonicalized_nosvgo_len2048.csv`
- `/kaggle/input/qwen25-coder-1p5b-instruct/`

Fixed Kaggle outputs:
- `/kaggle/working/adapter/`
- `/kaggle/working/checkpoints/`
- `/kaggle/working/submission.csv`
- `/kaggle/working/submission_debug.csv`


In [ ]:
from pathlib import Path

DATA_ROOT = Path('/kaggle/input/svg-kaggle-data')
BASE_MODEL_DIR = Path('/kaggle/input/qwen25-coder-1p5b-instruct')
TEST_CSV = DATA_ROOT / 'test.csv'
SAMPLE_SUBMISSION_CSV = DATA_ROOT / 'sample_submission.csv'
CLEAN_DATASET_CSV = DATA_ROOT / 'datasets' / 'train_canonicalized_nosvgo_len2048.csv'

WORK_ROOT = Path('/kaggle/working')
CHECKPOINT_DIR = WORK_ROOT / 'checkpoints'
ADAPTER_DIR = WORK_ROOT / 'adapter'
SUBMISSION_CSV = WORK_ROOT / 'submission.csv'
DEBUG_CSV = WORK_ROOT / 'submission_debug.csv'

TRAIN_MAX_LENGTH = 2048
SVG_MAX_NEW_TOKENS = 1536
SVG_BATCH_SIZE = 128
RENDER_SIZE = 256
MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256
STRICT_VIEWBOX = f'0 0 {RENDER_SIZE} {RENDER_SIZE}'
STRICT_BOX = (0.0, 0.0, float(RENDER_SIZE), float(RENDER_SIZE))
PREVIEW_SEED = 98748

required_paths = [TEST_CSV, SAMPLE_SUBMISSION_CSV, CLEAN_DATASET_CSV, BASE_MODEL_DIR]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing Kaggle inputs: {missing_paths}')

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print('Offline Kaggle paths ready:')
print(f'  BASE_MODEL_DIR: {BASE_MODEL_DIR}')
print(f'  CLEAN_DATASET_CSV: {CLEAN_DATASET_CSV}')
print(f'  TEST_CSV: {TEST_CSV}')
print(f'  SAMPLE_SUBMISSION_CSV: {SAMPLE_SUBMISSION_CSV}')
print(f'  CHECKPOINT_DIR: {CHECKPOINT_DIR}')
print(f'  ADAPTER_DIR: {ADAPTER_DIR}')
print(f'  SUBMISSION_CSV: {SUBMISSION_CSV}')
print(f'  DEBUG_CSV: {DEBUG_CSV}')


## Imports And Runtime Checks

This cell imports the required packages, prints the runtime environment, and fails early if the Kaggle notebook is missing a required dependency.


In [ ]:
import copy
import csv
import gc
import io
import re
import xml.etree.ElementTree as ET
from html import escape

import cairosvg
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import HTML, display
from lxml import etree
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Offline Base Model Load

This cell loads the offline base model snapshot from the fixed Kaggle input path with a 4-bit QLoRA-ready setup.


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)
model = prepare_model_for_kbit_training(model)

print(f"Model loaded from: {BASE_MODEL_DIR}")
print(f"Model dtype: {model.dtype}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")


## LoRA Setup And Trainer Config

This cell configures the LoRA adapter and the trainer settings for the fine-tuning run.


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

sft_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=50,
    save_strategy='steps',
    save_steps=1000,
    eval_strategy='no',
    optim='paged_adamw_8bit',
    gradient_checkpointing=False,
    max_grad_norm=0.3,
    report_to='none',
    max_length=TRAIN_MAX_LENGTH,
    packing=False,
)

print('Training config ready.')
print(sft_args)


## Dataset Prep And SVG-Only Loss Masking

This cell validates the cleaned CSV, formats each training sample as:

```text
Prompt: {prompt}
SVG:
{svg}
```

It then masks loss on everything before the literal boundary `SVG:\n`.


In [ ]:
with CLEAN_DATASET_CSV.open("r", encoding="utf-8", newline="") as handle:
    reader = csv.reader(handle)
    column_names = next(reader, None)
    if column_names != ["prompt", "svg"]:
        raise ValueError(f"Expected CSV columns ['prompt', 'svg'], found {column_names}")
    row_count = sum(1 for _ in reader)

raw_dataset = load_dataset("csv", data_files=str(CLEAN_DATASET_CSV))["train"]

# Format one training row into the prompt-to-SVG text used by the trainer.
def format_svg_sample(prompt: str, svg_code: str) -> str:
    return f"Prompt: {prompt}\nSVG:\n{svg_code}"

# Drop rows with missing prompt or SVG text.
def is_valid_row(example):
    return (
        example.get("prompt") is not None
        and example.get("svg") is not None
        and str(example["prompt"]).strip() != ""
        and str(example["svg"]).strip() != ""
    )

# Convert the cleaned CSV rows into the single text field expected by SFTTrainer.
def to_training_text(example):
    return {
        "text": format_svg_sample(
            prompt=str(example["prompt"]).strip(),
            svg_code=str(example["svg"]).strip(),
        )
    }

# Find the SVG boundary token sequence inside one tokenized sample.
def find_subsequence(sequence, subsequence):
    max_start = len(sequence) - len(subsequence)
    for start in range(max_start + 1):
        if sequence[start:start + len(subsequence)] == subsequence:
            return start
    return -1

# Mask loss on the prompt and keep loss only on the SVG tokens.
class SvgOnlyLossCollator:
    def __init__(self, tokenizer, response_template: str, max_length: int):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def __call__(self, features):
        if "input_ids" in features[0]:
            batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        else:
            texts = [feature["text"] for feature in features]
            batch = self.tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )

        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100

        for row_idx, input_ids in enumerate(batch["input_ids"].tolist()):
            start = find_subsequence(input_ids, self.response_template_ids)
            if start == -1:
                raise ValueError("Could not find the SVG response template in a training sample.")
            labels[row_idx, : start + len(self.response_template_ids)] = -100

        batch["labels"] = labels
        return batch

dataset = raw_dataset.filter(is_valid_row)
dataset = dataset.map(to_training_text, remove_columns=raw_dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]
svg_only_loss_collator = SvgOnlyLossCollator(tokenizer, response_template="SVG:\n", max_length=sft_args.max_length)

print(f"Clean dataset rows: {row_count}")
print(train_dataset[0]["text"][:1000])
assert "SVG:\n" in train_dataset[0]["text"]
print("SVG-only loss masking enabled after the response template 'SVG:\\n'.")

sample_count = min(100, len(train_dataset))
lengths = [len(tokenizer(train_dataset[i]["text"])["input_ids"]) for i in range(sample_count)]
print("Max token length:", max(lengths))
print("Avg token length:", sum(lengths) / len(lengths))


## Train And Save The Adapter

This cell runs the unchanged one-epoch training pass and saves the LoRA adapter into `/kaggle/working/adapter`.


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=svg_only_loss_collator,
)

trainer.train()
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")
print(f"Trainer global step: {trainer.state.global_step}")


## Reload The Base Model And Merge For Inference

This cell clears the training model, reloads the offline base model, applies the saved adapter, and merges it for deterministic inference.


In [ ]:
del trainer
try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Reload the tokenizer for inference and left-pad batched generation inputs.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    device_map="auto",
    torch_dtype=torch.float16,
    local_files_only=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model = model.merge_and_unload()
model.eval()

print("Merged adapter into a fresh base model for inference.")


## Strict SVG Cleanup And Validation

This cell keeps the submission path strict and lean. It extracts SVG from raw generation text, repairs malformed markup when possible, normalizes every output to the exact `256x256` contract, validates it, and falls back only when recovery fails.


In [ ]:
ET.register_namespace("", "http://www.w3.org/2000/svg")

SVG_NS = "http://www.w3.org/2000/svg"
STRICT_SVG_OPEN = f'<svg xmlns="{SVG_NS}" width="{RENDER_SIZE}" height="{RENDER_SIZE}" viewBox="{STRICT_VIEWBOX}">'
FALLBACK_SVG = f'{STRICT_SVG_OPEN}<rect width="{RENDER_SIZE}" height="{RENDER_SIZE}" fill="white"/></svg>'
DIRECT_ROOT_TAGS = {"defs", "title", "desc", "style"}
DRAWABLE_TAGS = {"path", "rect", "circle", "ellipse", "line", "polyline", "polygon", "use", "text", "image"}
DEBUG_COLUMNS = [
    "id",
    "reason",
    "gate_reason",
    "source_gate_reason",
    "normalized_gate_reason",
    "normalization_status",
    "strict_contract",
    "strict_issues",
    "collapsed",
    "hit_token_cap",
    "generated_tokens",
    "failure_reasons",
    "extracted_svg",
    "final_svg",
]
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan",
    "title", "desc", "style", "pattern", "marker", "filter",
}
EVENT_HANDLER_RE = re.compile(r"\son[a-zA-Z]+\s*=", re.IGNORECASE)
EXTERNAL_HREF_RE = re.compile(r"\s(?:href|xlink:href)\s*=\s*[\"']\s*(?:https?:|//)", re.IGNORECASE)
REMOTE_REF_RE = re.compile(r"@import\b|url\(\s*[\"']?(?:https?:|//)", re.IGNORECASE)
ROOT_TAG_RE = re.compile(r"<svg\b[^>]*>", flags=re.IGNORECASE | re.DOTALL)

# Remove the namespace prefix from a parsed XML tag.
def strip_namespace(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag

# Pull the SVG block out of raw model text.
def extract_svg(text: str) -> str:
    text = str(text or "").strip()
    match = re.search(r"<svg\b[^>]*>.*?</svg>", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()
    if "SVG:" in text:
        text = text.split("SVG:", 1)[1].strip()
    start = text.find("<svg")
    if start != -1:
        return text[start:].strip()
    return text

# Render one SVG to catch non-renderable outputs.
def render_svg(svg: str, size: int = RENDER_SIZE):
    try:
        png = cairosvg.svg2png(bytestring=svg.encode("utf-8"), output_width=size, output_height=size)
        return np.array(Image.open(io.BytesIO(png)).convert("RGB"))
    except Exception:
        return None

# Read one attribute value from the root SVG opening tag.
def get_attr_value(opening_tag: str, attr_name: str):
    pattern = rf"(\s{re.escape(attr_name)}\s*=\s*)([\"'])(.*?)\2"
    match = re.search(pattern, opening_tag, flags=re.IGNORECASE | re.DOTALL)
    return None if match is None else match.group(3)

# Format a float compactly for matrix transforms.
def format_number(value: float) -> str:
    if abs(value - round(value)) < 1e-9:
        return str(int(round(value)))
    text = f"{value:.12g}"
    return text.rstrip("0").rstrip(".") if "e" not in text and "." in text else text

# Parse a numeric width or height attribute.
def parse_numeric_attr(value):
    if value is None:
        return None
    match = re.fullmatch(r"\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)\s*(px)?\s*", str(value))
    if match is None:
        return None
    numeric = float(match.group(1))
    return numeric if numeric > 0 else None

# Parse an SVG viewBox into four floats.
def parse_viewbox(value):
    if value is None:
        return None
    pieces = [piece for piece in re.split(r"[\s,]+", str(value).strip()) if piece]
    if len(pieces) != 4:
        return None
    try:
        x, y, width, height = [float(piece) for piece in pieces]
    except ValueError:
        return None
    return (x, y, width, height) if width > 0 and height > 0 else None

# Determine the source coordinate box before strict 256x256 normalization.
def derive_source_box(root: ET.Element):
    viewbox = parse_viewbox(root.attrib.get("viewBox"))
    if viewbox is not None:
        return viewbox, "viewBox"
    width = parse_numeric_attr(root.attrib.get("width"))
    height = parse_numeric_attr(root.attrib.get("height"))
    if width is not None and height is not None:
        return (0.0, 0.0, width, height), "width_height"
    return STRICT_BOX, "default"

# Rebuild the SVG root to the exact competition wrapper and preserve geometry with a transform when needed.
def rebuild_svg_root(root: ET.Element, source_box, source_kind: str):
    new_root = ET.Element(f"{{{SVG_NS}}}svg")
    new_root.set("width", str(RENDER_SIZE))
    new_root.set("height", str(RENDER_SIZE))
    new_root.set("viewBox", STRICT_VIEWBOX)

    source_x, source_y, source_width, source_height = source_box
    needs_transform = source_box != STRICT_BOX
    transform_group = None

    if needs_transform:
        sx = RENDER_SIZE / source_width
        sy = RENDER_SIZE / source_height
        tx = -source_x * sx
        ty = -source_y * sy
        transform_group = ET.Element(f"{{{SVG_NS}}}g")
        transform_group.set(
            "transform",
            f"matrix({format_number(sx)} 0 0 {format_number(sy)} {format_number(tx)} {format_number(ty)})",
        )

    group_inserted = False
    for child in list(root):
        child_copy = copy.deepcopy(child)
        child_tag = strip_namespace(child.tag)
        if child_tag in DIRECT_ROOT_TAGS or transform_group is None:
            new_root.append(child_copy)
            continue
        if not group_inserted:
            new_root.append(transform_group)
            group_inserted = True
        transform_group.append(child_copy)

    status = "unchanged" if not needs_transform else f"scaled_from_{source_kind}"
    return ET.tostring(new_root, encoding="unicode"), status

# Canonicalize one SVG to the exact 256x256 root contract.
def canonicalize_to_strict_256(svg: str):
    try:
        root = ET.fromstring(svg)
    except Exception:
        return svg, "parse_failed"
    if strip_namespace(root.tag) != "svg":
        return svg, "root_not_svg"
    source_box, source_kind = derive_source_box(root)
    return rebuild_svg_root(root, source_box, source_kind)

# Check whether one SVG root matches the exact final submission contract.
def strict_contract_issues(svg: str) -> list[str]:
    issues = []
    opening_match = ROOT_TAG_RE.search(svg)
    if opening_match is None:
        return ["strict_parse_failed"]
    opening_tag = opening_match.group(0)
    if get_attr_value(opening_tag, "xmlns") != SVG_NS:
        issues.append("missing_xmlns")
    try:
        root = ET.fromstring(svg)
    except Exception:
        return issues + ["strict_parse_failed"]
    if strip_namespace(root.tag) != "svg":
        return issues + ["root_not_svg"]
    if root.attrib.get("width") != str(RENDER_SIZE) or root.attrib.get("height") != str(RENDER_SIZE):
        issues.append("width_height_not_exact")
    if root.attrib.get("viewBox") != STRICT_VIEWBOX:
        issues.append("viewbox_not_exact")
    return issues

# Run the lightweight submission validity gate before strict contract checks.
def validity_gate(svg: str):
    if not isinstance(svg, str) or not svg.strip():
        return 0, "svg_too_long_or_empty"
    svg = svg.strip()
    if len(svg) > MAX_SVG_LENGTH:
        return 0, "svg_too_long_or_empty"
    if EVENT_HANDLER_RE.search(svg):
        return 0, "disallowed_attr:event_handler"
    if EXTERNAL_HREF_RE.search(svg):
        return 0, "disallowed_ref:external_href"
    if REMOTE_REF_RE.search(svg):
        return 0, "disallowed_ref:remote_url"
    try:
        root = ET.fromstring(svg)
    except Exception:
        return 0, "parse_failed"
    if strip_namespace(root.tag) != "svg":
        return 0, "root_not_svg"
    path_count = 0
    for elem in root.iter():
        tag = strip_namespace(elem.tag)
        if tag not in ALLOWED_TAGS:
            return 0, f"disallowed_tag:{tag}"
        if tag == "path":
            path_count += 1
    if path_count > MAX_PATH_COUNT:
        return 0, "too_many_paths"
    if render_svg(svg) is None:
        return 0, "render_failed"
    return 1, "valid"

# Trim trailing junk and close a missing root tag when the model stops early.
def repair_svg(svg: str) -> str:
    if not svg:
        return svg
    svg = svg.strip()
    start = svg.find("<svg")
    if start != -1:
        svg = svg[start:]
    if "SVG:" in svg:
        svg = svg.split("SVG:", 1)[-1].strip()
    if "</svg>" in svg:
        end = svg.rfind("</svg>")
        svg = svg[: end + len("</svg>")]
    if "<svg" in svg and "</svg>" not in svg:
        svg += "</svg>"
    svg = re.sub(r"[A-Za-z0-9.\-]+$", "", svg)
    svg = re.sub(r"<[^>]*$", "", svg)
    return svg

# Let lxml recover one malformed SVG when a simple string repair is not enough.
def recover_svg_with_lxml(svg: str) -> str:
    if not svg or "<svg" not in svg:
        return svg
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(svg.encode("utf-8"), parser=parser)
        return svg if root is None else etree.tostring(root, encoding="unicode")
    except Exception:
        return svg

# Reject empty or obviously collapsed drawings.
def looks_collapsed(svg: str) -> bool:
    try:
        root = ET.fromstring(svg)
    except Exception:
        return True
    drawables = [elem for elem in root.iter() if strip_namespace(elem.tag) in DRAWABLE_TAGS]
    if not drawables:
        return True
    path_drawables = [elem for elem in drawables if strip_namespace(elem.tag) == "path"]
    if path_drawables and all(not elem.attrib.get("d", "").strip() for elem in path_drawables):
        if not [elem for elem in drawables if strip_namespace(elem.tag) != "path"]:
            return True
    return sum(1 for _ in root.iter()) <= 1

# Build one accepted candidate after gate, normalization, and strict checks succeed.
def candidate_from_svg(extracted_svg: str, stage_svg: str, reason: str):
    source_valid, source_gate_reason = validity_gate(stage_svg)
    if source_valid == 0:
        return None, source_gate_reason
    final_svg, normalization_status = canonicalize_to_strict_256(stage_svg)
    normalized_valid, normalized_gate_reason = validity_gate(final_svg)
    if normalized_valid == 0:
        return None, normalized_gate_reason
    strict_issues = strict_contract_issues(final_svg)
    if strict_issues or looks_collapsed(final_svg):
        return None, "strict_contract_failed" if strict_issues else "collapsed"
    return {
        "reason": reason,
        "gate_reason": normalized_gate_reason,
        "source_gate_reason": source_gate_reason,
        "normalized_gate_reason": normalized_gate_reason,
        "normalization_status": normalization_status,
        "strict_contract": True,
        "strict_issues": "",
        "collapsed": False,
        "hit_token_cap": False,
        "generated_tokens": 0,
        "failure_reasons": "",
        "extracted_svg": extracted_svg,
        "final_svg": final_svg,
    }, normalized_gate_reason

# Turn raw model text into one strict final SVG or a valid fallback.
def clean_svg_output(raw_text: str):
    extracted_svg = extract_svg(str(raw_text or ""))
    failures = []

    raw_candidate, raw_failure = candidate_from_svg(extracted_svg, extracted_svg, "valid")
    if raw_candidate is not None:
        return raw_candidate
    failures.append(f"raw={raw_failure}")

    repaired_svg = repair_svg(extracted_svg)
    repaired_candidate, repaired_failure = candidate_from_svg(extracted_svg, repaired_svg, "repaired")
    if repaired_candidate is not None:
        repaired_candidate["failure_reasons"] = "|".join(failures)
        return repaired_candidate
    failures.append(f"repaired={repaired_failure}")

    xml_svg = recover_svg_with_lxml(repaired_svg)
    xml_candidate, xml_failure = candidate_from_svg(extracted_svg, xml_svg, "xml")
    if xml_candidate is not None:
        xml_candidate["failure_reasons"] = "|".join(failures)
        return xml_candidate
    failures.append(f"xml={xml_failure}")

    fallback_issues = strict_contract_issues(FALLBACK_SVG)
    return {
        "reason": "fallback",
        "gate_reason": failures[-1].split("=", 1)[-1] if failures else "fallback",
        "source_gate_reason": "",
        "normalized_gate_reason": "",
        "normalization_status": "fallback",
        "strict_contract": not fallback_issues,
        "strict_issues": ",".join(fallback_issues),
        "collapsed": False,
        "hit_token_cap": False,
        "generated_tokens": 0,
        "failure_reasons": "|".join(failures),
        "extracted_svg": extracted_svg,
        "final_svg": FALLBACK_SVG,
    }

fallback_strict_issues = strict_contract_issues(FALLBACK_SVG)
if fallback_strict_issues:
    raise ValueError(f"Fallback SVG is not strict-contract compliant: {fallback_strict_issues}")

strict_result = clean_svg_output(
    '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><circle cx="128" cy="128" r="64"/></svg>'
)
assert strict_result["reason"] == "valid"
assert strict_contract_issues(strict_result["final_svg"]) == []


## Build The Submission

This cell runs deterministic one-pass inference, cleans every output into the strict submission contract, and writes both `submission.csv` and `submission_debug.csv`.


In [ ]:
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_CSV, keep_default_na=False)
test_df = pd.read_csv(TEST_CSV, keep_default_na=False)
test_df = test_df.dropna(subset=["id", "prompt"]).copy()
test_df["id"] = test_df["id"].astype(str)
test_df["prompt"] = test_df["prompt"].astype(str)
sample_submission_df["id"] = sample_submission_df["id"].astype(str)

if sample_submission_df.columns.tolist() != ["id", "svg"]:
    raise ValueError(f"Expected sample submission columns ['id', 'svg'], found {sample_submission_df.columns.tolist()}")
if sample_submission_df["id"].tolist() != test_df["id"].tolist():
    raise AssertionError("sample_submission.csv ids do not match test.csv ids in order.")

# Normalize mixed bool-like values from debug columns.
def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return value != 0
    return str(value).strip().lower() in {"true", "1", "yes", "y"}

# Build the exact plain prompt prefix used during training and inference.
def build_svg_input(prompt: str) -> str:
    return f"Prompt: {prompt.strip()}\nSVG:\n"

# Generate batched text with a small OOM backoff path for smaller Kaggle GPUs.
def generate_text_batch(inputs_text, batch_size, max_new_tokens):
    batch_outputs = []
    next_index = 0
    current_batch_size = max(1, batch_size)
    progress_bar = tqdm(total=len(inputs_text))

    while next_index < len(inputs_text):
        batch_inputs_text = inputs_text[next_index:next_index + current_batch_size]
        inputs = None
        outputs = None
        try:
            inputs = tokenizer(
                batch_inputs_text,
                return_tensors="pt",
                padding=True,
                truncation=True,
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            input_width = int(inputs["input_ids"].shape[1])
            generated_only = outputs[:, input_width:].detach().cpu()
            decoded_batch = tokenizer.batch_decode(generated_only, skip_special_tokens=True)
            eos_token_id = tokenizer.eos_token_id

            for raw_text, generated_ids in zip(decoded_batch, generated_only):
                generated_token_ids = generated_ids.tolist()
                if eos_token_id is not None and eos_token_id in generated_token_ids:
                    eos_index = generated_token_ids.index(eos_token_id)
                    effective_token_ids = generated_token_ids[:eos_index]
                    hit_token_cap = False
                else:
                    effective_token_ids = generated_token_ids
                    hit_token_cap = len(effective_token_ids) >= max_new_tokens
                batch_outputs.append(
                    {
                        "raw_text": raw_text,
                        "generated_tokens": len(effective_token_ids),
                        "hit_token_cap": hit_token_cap,
                    }
                )

            next_index += len(batch_inputs_text)
            progress_bar.update(len(batch_inputs_text))
        except torch.cuda.OutOfMemoryError:
            if current_batch_size == 1:
                raise
            current_batch_size = max(1, current_batch_size // 2)
            print(f"CUDA OOM. Retrying with batch size {current_batch_size}.")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        finally:
            del inputs
            del outputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    progress_bar.close()
    return batch_outputs

# Clean one generation result and attach token-cap metadata.
def build_candidate(generation_output: dict):
    candidate = clean_svg_output(generation_output.get("raw_text", ""))
    candidate["generated_tokens"] = int(generation_output.get("generated_tokens", 0))
    candidate["hit_token_cap"] = as_bool(generation_output.get("hit_token_cap", False))
    return candidate

# Run deterministic inference and build the submission/debug DataFrames.
def build_submission(test_df, svg_batch_size=SVG_BATCH_SIZE):
    prompts = test_df["prompt"].astype(str).tolist()
    generation_outputs = generate_text_batch(
        [build_svg_input(prompt) for prompt in prompts],
        batch_size=svg_batch_size,
        max_new_tokens=SVG_MAX_NEW_TOKENS,
    )
    candidates = [build_candidate(output) for output in generation_outputs]

    submission_df = pd.DataFrame(
        {
            "id": test_df["id"].tolist(),
            "svg": [candidate["final_svg"] for candidate in candidates],
        }
    )

    debug_rows = []
    for row, candidate in zip(test_df.itertuples(index=False), candidates):
        debug_row = {column: candidate.get(column, "") for column in DEBUG_COLUMNS if column != "id"}
        debug_row["id"] = row.id
        debug_rows.append(debug_row)
    debug_df = pd.DataFrame(debug_rows)[DEBUG_COLUMNS]
    return submission_df, debug_df

submission_df, debug_df = build_submission(test_df)
submission_df.to_csv(SUBMISSION_CSV, index=False)
debug_df.to_csv(DEBUG_CSV, index=False)

print("Final reason counts:")
print(debug_df["reason"].value_counts())
print(f"Final strict-contract pass rate: {float(debug_df['strict_contract'].map(as_bool).mean()):.4f}")
print(f"Token-cap hits: {int(debug_df['hit_token_cap'].map(as_bool).sum())}")
display(submission_df.head())
display(debug_df.head())


## Final Checks And Preview

This cell reloads the saved CSVs, verifies the final contract, and renders 20 random submission rows with prompts for a quick visual smoke test.


In [ ]:
saved_submission_df = pd.read_csv(SUBMISSION_CSV, keep_default_na=False)
saved_debug_df = pd.read_csv(DEBUG_CSV, keep_default_na=False)
prompt_df = pd.read_csv(TEST_CSV, keep_default_na=False)[["id", "prompt"]].copy()
prompt_df["id"] = prompt_df["id"].astype(str)
saved_submission_df["id"] = saved_submission_df["id"].astype(str)

if saved_submission_df.columns.tolist() != ["id", "svg"]:
    raise ValueError(f"Expected submission columns ['id', 'svg'], found {saved_submission_df.columns.tolist()}")
if len(saved_submission_df) != len(sample_submission_df):
    raise AssertionError(f"Expected {len(sample_submission_df)} submission rows, found {len(saved_submission_df)}")

strict_failures = [svg for svg in saved_submission_df["svg"].tolist() if strict_contract_issues(svg)]
if strict_failures:
    raise AssertionError(f"Found {len(strict_failures)} non-strict SVG outputs in submission.csv")

print(f"Saved submission rows: {len(saved_submission_df)}")
print(f"Saved debug rows: {len(saved_debug_df)}")
print("Saved debug columns:")
print(saved_debug_df.columns.tolist())
print("Saved reason counts:")
print(saved_debug_df["reason"].value_counts())

display(saved_submission_df.head())
display(saved_debug_df.head())

preview_df = saved_submission_df.merge(prompt_df, on="id", how="left")
sample_count = min(20, len(preview_df))
sampled_preview_df = preview_df.sample(n=sample_count, random_state=PREVIEW_SEED).reset_index(drop=True)

cards = []
for row in sampled_preview_df.itertuples(index=False):
    cards.append(
        '<div style="border:1px solid #ddd;border-radius:8px;padding:12px;background:white;">'
        f'<div style="font-family:monospace;font-size:12px;margin-bottom:8px;">id: {escape(str(row.id))}</div>'
        f'<div style="font-size:12px;line-height:1.4;margin-bottom:8px;color:#333;">prompt: {escape(str(row.prompt))}</div>'
        f'<div style="width:256px;height:256px;border:1px solid #eee;background:#fff;overflow:hidden;">{row.svg}</div>'
        '</div>'
    )

grid_html = (
    '<div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(280px, 1fr));gap:12px;">'
    + ''.join(cards)
    + '</div>'
)

print(f"Previewing {sample_count} random rows from {SUBMISSION_CSV} with seed {PREVIEW_SEED}")
display(HTML(grid_html))
